In [1]:
import numpy as np

np.set_printoptions(precision=3,suppress=True)

In [2]:
import sys
from os.path import dirname

sys.path.append(dirname("../src/"))

In [3]:
from samosa.symmetry.point_group_utils import point_group

# Index

* [Brief description of the `Group` object properties](#intro_group)
* [3D point groups (conceptual)](#pg_intro)
    - [Axial point groups](#pg_intro_axial)
    - [Polyhedral point groups](#pg_intro_polyhedral)
    - [More information on 3D point groups](#pg_intro_more)
* [Initializing a point group](#pg_init)
* [Basic properties](#pg_basics)
    - [Predefined properties (generators, name, order)](#pg_basics_predefined)
    - [Group elements](#pg_basics_elements)
    - [Conjugate classes](#pg_basics_classes)
* [Representation theory](#pg_representations)
    - [Character table](#pg_representations_characters)
    - [Irreducible representations (irreps)](#pg_representations_irreps)
* [Orbits and stabilizers](#pg_orbit)
    - [Conceptual introduction](#pg_orbit_intro)
    - [Calculating an orbit of a 3D point](#pg_orbit_calculation)
    - [Changing the basis of the group to permutations of orbit elements](#pg_orbit_permutations)

# Brief description of the `Group` object properties <a name="intro_group"></a>

Mathematical description of symmetry is typically done using the notion of a __group__.
Formally, a group $\mathcal{G}$ is defined by a (non-empty) set of elements and a binary operation (group element product), which satisfy three requirements:

1. The product of group elements is __associative__, _i.e._ $A\cdot(B\cdot C) = (A\cdot B)\cdot C$, for $A$, $B$, $C$ $\in\mathcal{G}$;
2. There is an __identity element__ $E\in\mathcal{G}$, which satisfies $A\cdot E = E\cdot A = A$ for all $A\in\mathcal{G}$;
3. For each element $A\in\mathcal{G}$, there is an __inverse element__ $A^{-1}\in\mathcal{G}$, such that $A\cdot A^{-1} = A^{-1}\cdot A = E$.

It can be shown that the set of all symmetries of a given object forms a group if we take the symmetry action as the group element product.
In the `samosa` module, the `Group` object serves as a container for the main properties of a mathematical group. 

This notebook introduces the special category of groups most relevant to physics applications: __3D point groups__. 
For a tutorial on defining a `Group` object "from scratch", please refer to [custom_group.ipynb](./custom_group.ipynb).

# 3D point groups (conceptual) <a name="pg_intro"></a>

As per their name, point groups describe the symmetry of a single point fixed in space.
In 3D, these groups derrive from the symmetry of a sphere, O(3), and consist of proper/improper rotations, inversion, and mirror reflections (see [group_elements.ipynb](./group_elements.ipynb) for some examples).
As such, there are two main categories of 3D point groups: __axial__ (7 types of groups) and __polyhedral__ (7 groups) point groups.

## Axial point groups <a name="pg_intro_axial"></a>

Axial point groups characterize objects with a prism-like symmetry.
Typically, such objects contain a primary $n$-fold rotation axis and, optionally, secondary 2-fold rotation axes or mirror planes.
Since $n$ can be any integer, there is an infinite number of such groups.
However, depending on the nature of the secondary symmetry elements, axial point groups belong to one of the 7 types ([Schoenflies notation](https://en.wikipedia.org/wiki/Schoenflies_notation) will be used throughout for definitions of the point group symbols.):

* 4 groups without secondary rotation axes (cyclic groups $\mathrm{C}_n$, $\mathrm{C}_{nv}$, $\mathrm{C}_{nh}$, $\mathrm{S}_n$);
* 3 groups with secondary 2-fold rotations (dihedral groups $\mathrm{D}_n$, $\mathrm{D}_{nd}$, $\mathrm{D}_{nh}$).

In the standard notation, $z$-axis ($[001]$) is selected as the $n$-fold axis.
Where necessary `samosa` uses the $y$-axis ($[010]$) as the secondary rotation/reflection axis.

### $\mathrm{C}_n$ groups

Are generated by a single $n$-fold proper rotation. 
As a result, these groups are chiral and Abelian (all symmetry elements commute).

### $\mathrm{C}_{nv}$ groups

Are generated by $n$-fold proper rotation with $n>1$ and a reflection with a normal vector of the mirror plane perpendicular to the $n$-fold axis.

### $\mathrm{C}_{nh}$ groups

Are generated by $n$-fold proper rotation and a reflection with a normal vector of the mirror plane parallel to the $n$-fold axis.

### $\mathrm{S}_n$ groups

Are generated by a single $n$-fold improper rotation, where $n$ is even.
These groups are chiral and Abelian (all symmetry elements commute).

### $\mathrm{D}_n$ groups

Are generated by $n$-fold proper rotation with $n>1$ and a secondary 2-fold rotation axis perpendicular to the $n$-fold axis.

### $\mathrm{D}_{nd}$ groups

Are generated by $2n$-fold improper rotation with $n>1$ and a reflection with a normal vector of the mirror plane perpendicular to the $n$-fold axis.

### $\mathrm{D}_{nh}$ groups

Are generated by $n$-fold proper rotation with $n>1$ and two reflections with a normal vectors of the mirror planes correspondingly parallel and perpendicular to the $n$-fold axis.

## Polyhedral point groups <a name="pg_intro_polyhedral"></a>

The remaining 7 point groups describe the symmetries of common polyhedra:

* Tetrahedral point groups ($\mathrm{T}$, $\mathrm{T}_d$, $\mathrm{T}_h$);
* Octahedral point groups ($\mathrm{O}$, $\mathrm{O}_h$);
* Icosahedral point groups ($\mathrm{I}$, $\mathrm{I}_h$).

These groups are characterized by multiple $n$-fold ($n>2$) rotation axes.

As per the standard notation, $z$-axis ($[001]$) is typically selected as the axis of highest symmetry (largest value of $n$).
The descriptions below also indicate the axes used for the generators of the point groups.

### $\mathrm{T}$ group

Is the (chiral) rotational symmetry group of a [regular tetrahedron](https://en.wikipedia.org/wiki/Tetrahedron#Regular_tetrahedron).
It is generated by a 3-fold proper rotation around $[111]$ axis, and a 2-fold proper rotation around $[001]$ axis.

### $\mathrm{T}_d$ group

Decribes the symmetry of a regular tetrahedron.
It is generated by two 4-fold improper rotations (around $[001]$ and $[100]$ axes, respectively).

### $\mathrm{T}_h$ group

Is the symmetry of a [pyritohedron](https://en.wikipedia.org/wiki/Pyritohedron) (also a [volleyball ball](https://en.wikipedia.org/wiki/Volleyball_(ball))).
It is generated by a 3-fold proper rotation and a reflection with the mirror plane perpendicular to the $[001]$ axis.

### $\mathrm{O}$ group

Is the rotational group of an [octahedron](https://en.wikipedia.org/wiki/Octahedron).
It is generated by two perpendicular 4-fold proper rotations (with axes taken along $[001]$ and $[100$ axes, respectively).

### $\mathrm{O}_h$ group

Describes the full symmetry of an octahedron.
It is generated by a 6-fold improper rotation around $[111]$ axis and a 4-fold improper rotation around $[001]$ axis.

### $\mathrm{I}$ group

Is the group of rotations of an [icosahedron](https://en.wikipedia.org/wiki/Icosahedron) and [dodecahedron](https://en.wikipedia.org/wiki/Dodecahedron).
It is generated by two proper 5-fold rotations with axes along $[01\phi]$ and $[\phi01]$, where $\phi = \frac{1+\sqrt{5}}{2}$ is the Golden ratio.

### $\mathrm{I}_h$ group

Describes the full symmetry of an icosahedron and dodecahedron.
It is generated by two improper 10-fold rotations with axes along $[01\phi]$ and $[\phi01]$, where $\phi = \frac{1+\sqrt{5}}{2}$ is the Golden ratio.


## More information on 3D point groups <a name="pg_intro_more"></a>

[Gernot Katzer's website](http://gernot-katzers-spice-pages.com/character_tables/index.html) provides a lot of useful information about the 3D point groups.

# Initializing a point group <a name="pg_init"></a>

In [4]:
"""
3D point groups are initialized as a Group objects using point_group function
by entering the correct Schoenflies symbol.
"""

# Initialize a cyclic group of order 6 
c6_group = point_group('C6')

print(c6_group)

Symmetry group C6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]];

Group order: 6;


In [5]:
"""
While the [001] axis is chosen by default for the n-fold axes (or,
equivalently, the highest symmetry axes). However, we can specify the primary
and secondary symmetry axes after the Schoenflies symbol. 
"""

# Initialize a cyclic group of order 6 with 6-fold axis along [100] 
c6_x_group = point_group('C6', [1,0,0])

print(c6_x_group)

Symmetry group C6

Generators:
[[ 1.     0.     0.   ]
 [ 0.     0.5   -0.866]
 [ 0.     0.866  0.5  ]];

Group order: 6;


In [6]:
# Initialize C6v group with 6-fold axis along [100] and the secondary axis
# (mirror normal) along [010]
c6v_x_group = point_group('C6v', [1,0,0], [0,1,0])

print(c6v_x_group)

Symmetry group C6v

Generators:
[[ 1.     0.     0.   ]
 [ 0.     0.5   -0.866]
 [ 0.     0.866  0.5  ]]

[[ 1. -0. -0.]
 [-0. -1. -0.]
 [ 0. -0.  1.]];

Group order: 12;


In [7]:
# It is important to specify the right number of axes. The code will return an
# error if we specify an incorrect number of axes.


# Uncomment to view the errors
#point_group('C6', [1,0,0], [0,1,0]) # Too many axes specified
#point_group('C6v', [1,0,0],) # Not enough axes specified

In [8]:
"""
As with the group elements, we can specify the basis of the generator matrices
using the `basis` argument.
"""

# Initialize a cyclic group of order 6 with generators in the hexagonal basis 
B = np.array([ [    1,            0, 0 ],
               [ -0.5, np.sqrt(3)/2, 0 ],
               [    0,            0, 1 ]])

c6_hex_group = point_group('C6', basis=B)

print(c6_hex_group)

Symmetry group C6

Generators:
[[ 1. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0.  1.]];

Group order: 6;


In [9]:
"""
Test out different point groups
"""

# Initialize another group
test_group = point_group('Oh') # try inserting a different symbol

print(test_group)

Symmetry group Oh

Generators:
[[-0. -1. -0.]
 [-0. -0. -1.]
 [-1. -0. -0.]]

[[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0. -1.]];

Group order: 48;


# Basic properties <a name="pg_basics"></a>

## Predefined properties (name, generators, order) <a name="pg_basics_predefined"></a> 

In [10]:
"""
samosa predefines a number of point group's properties, to avoid legthy 
calculations. The summary of these properties is displayed when we use the
print function.
"""

# Initialize another group
test_group = point_group('D6')

print(test_group)

print('\n\n')

print(f'Group name = {test_group.name}\n\n'
      f'Group generators = \n{test_group.generators}\n\n'
      f'Group order = {test_group.order}')

# Recall that args_calculated=False means that some of the properties of the 
# MatrixGroupElement objects have been defined via user input

Symmetry group D6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]];

Group order: 12;



Group name = D6

Group generators = 
[MatrixGroupElement(operator=array([[ 0.5  , -0.866,  0.   ],
       [ 0.866,  0.5  ,  0.   ],
       [ 0.   ,  0.   ,  1.   ]]), args_calculated=False), MatrixGroupElement(operator=array([[-1.,  0.,  0.],
       [ 0.,  1.,  0.],
       [-0.,  0., -1.]]), args_calculated=False)]

Group order = 12


## Group elements <a name="pg_basics_elements"></a>

In [11]:
"""
By default, the point groups are initialized without group elements, since
some groups may be too large and require a lot of memory to store them.
We can calculate all elements using `calculate_elements()` method.
"""

# Use store_data=True to record data to the `elements` property
elements_list = test_group.calculate_elements(store_data=True)

test_group.elements # Will return None if store_data=False

[MatrixGroupElement(operator=array([[ 0.5  , -0.866,  0.   ],
        [ 0.866,  0.5  ,  0.   ],
        [ 0.   ,  0.   ,  1.   ]]), args_calculated=False),
 MatrixGroupElement(operator=array([[-1.,  0.,  0.],
        [ 0.,  1.,  0.],
        [-0.,  0., -1.]]), args_calculated=False),
 MatrixGroupElement(operator=array([[-0.5  , -0.866,  0.   ],
        [ 0.866, -0.5  ,  0.   ],
        [ 0.   ,  0.   ,  1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[-0.5  ,  0.866,  0.   ],
        [ 0.866,  0.5  ,  0.   ],
        [ 0.   ,  0.   , -1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[-0.5  , -0.866,  0.   ],
        [-0.866,  0.5  ,  0.   ],
        [ 0.   ,  0.   , -1.   ]]), args_calculated=True),
 MatrixGroupElement(operator=array([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]]), args_calculated=True),
 MatrixGroupElement(operator=array([[-1.,  0.,  0.],
        [ 0., -1.,  0.],
        [ 0.,  0.,  1.]]), args_calculated=True),
 Ma

In [12]:
print(test_group)

Symmetry group D6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]];

Group order: 12;

Group elements:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]]

[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[-0.5   -0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

[[-1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]

[[ 0.5    0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5   -0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5    0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0. -1.]];


## Conjugate classes <a name="pg_basics_classes"></a>

In [13]:
"""
It is often useful to calculate conjugate classes of group elements. 
A conjugate class is a set consisting of elements 

C = {A^-1 B A}

for A and B in the same group.

By default, the point groups are initialized without conjugate_classes, since
some groups may be too large and require a lot of memory to store them.
We can calculate conjugate classes using `calculate_classes()` method.
"""

# Use store_data=True to record data to the `elements` property
elements_list, class_list = test_group.calculate_classes(store_data=True)

test_group.conjugate_classes # Will return None if store_data=False

{(0, 9): {'sign': 1.0, 'trace': 2.0},
 (1, 7, 8): {'sign': 1.0, 'trace': -1.0},
 (2, 10): {'sign': 1.0, 'trace': 0.0},
 (3, 11, 4): {'sign': 1.0, 'trace': -1.0},
 (5,): {'sign': 1.0, 'trace': 3.0},
 (6,): {'sign': 1.0, 'trace': -1.0}}

In [14]:
print(test_group)

Symmetry group D6

Generators:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]];

Group order: 12;

Group elements:
[[ 0.5   -0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-1.  0.  0.]
 [ 0.  1.  0.]
 [-0.  0. -1.]]

[[-0.5   -0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [ 0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[-0.5   -0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

[[-1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]

[[ 0.5    0.866  0.   ]
 [ 0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5   -0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.    -1.   ]]

[[ 0.5    0.866  0.   ]
 [-0.866  0.5    0.   ]
 [ 0.     0.     1.   ]]

[[-0.5    0.866  0.   ]
 [-0.866 -0.5    0.   ]
 [ 0.     0.     1.   ]]

[[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0. -1.]];

Conjugate classes:
{(0, 9): {'sign': 1.0,

# Representation theory <a name="pg_representations"></a>

## Character table <a name="pg_representations_characters"></a>

## Irreducible representations (irreps) <a name="pg_representations_irreps"></a>

# Orbits and stabilizers <a name="pg_orbit"></a>

## Conceptual introduction <a name="pg_orbit_intro"></a>

## Calculating an orbit of a 3D point <a name="pg_orbit_calculation"></a>

## Changing the basis of the group to permutations of orbit elements <a name="pg_orbit_permutations"></a>